In [ ]:
import pandas as pd

def read_csv(i):
    df = pd.read_csv(f'day__{i}.csv', index_col=0)
    df.insert(0, "day", i)
    df['dac_family'] = df['dac_family'].fillna('healthy')
    return df

df = pd.concat([read_csv(i) for i in range(10) ])

df[['dac_family','day']].value_counts().unstack()

values=df.to_numpy().tolist()
for i in [0,2,5,6,7,8,9]:
    values.append([i, "tofsee", 0,0,0,0,0,0,0,0,0])

df = pd.DataFrame(values, columns=df.columns).sort_values(by=['day', 'dac_family'])
df




,day,dac_family,qr,q,r,nx,valids,unique_dn,unique_valids,unique_nxd,unique_nxd_valids
0,0,conficker,21516,5082,16434,7794,21516,499,499,367,367
6,0,healthy,33075694,15044609,18031085,994122,32130786,343312,298250,72332,28574
1,0,modpack,1086,464,622,0,1086,1,1,0,0
2,0,necurs,1315664,315516,1000148,613894,1315664,14023,14023,13632,13632
3,0,pitou,5684,2181,3503,2354,5684,20,20,17,17
...,...,...,...,...,...,...,...,...,...,...,...
67,9,necurs,1776472,480507,1295965,875338,1776472,8084,8084,8037,8037
68,9,pitou,64494,15010,49484,37366,64494,33,33,33,33
72,9,suppobox,0,0,0,0,0,0,0,0,0
79,9,tofsee,0,0,0,0,0,0,0,0,0


In [4]:

import pandas as pd
import numpy as np
from sqlalchemy import create_engine


dbalchemy = create_engine(f"postgresql+psycopg2://postgres@localhost:5432/dns_mac")

def get(day):
    print(day)
    features = \
    """COUNT(*) AS QR,
	COUNT(*) filter (WHERE IS_R IS FALSE) AS Q,
	COUNT(*) filter (WHERE IS_R IS TRUE) AS R,
	COUNT(*) filter (WHERE RCODE=0) AS OK,
	COUNT(*) filter (WHERE RCODE=3) AS NX,
	COUNT(*) FILTER (WHERE NOT bigdn_m3.regex_check) AS NOTVALIDs,
	COUNT(DISTINCT DN_ID) AS UNIQUE_DN,
	COUNT(DISTINCT DN_ID) FILTER (WHERE bigdn_m3.regex_check) AS UNIQUE_VALIDS,
	COUNT(DISTINCT DN_ID) FILTER (WHERE RCODE=0) AS UNIQUE_OK,
	COUNT(DISTINCT DN_ID) FILTER (WHERE RCODE=3) AS UNIQUE_NXD,
	COUNT(DISTINCT DN_ID) FILTER (WHERE RCODE=3 AND NOT bigdn_m3.regex_check) AS UNIQUE_NXD_NOTVALIDS"""

    df = pd.read_sql(f"""
SELECT
    {day} as day,
	FLOOR(SECONDS/3600) as HOUR,
	CASE WHEN cardinality(bigdn_m3.dac_families) > 0 AND bigdn_m3.dac_count_between > 0 THEN bigdn_m3.dac_families[1] ELSE NULL::text END
	AS dac_family,
    {features}
FROM
	MESSAGE3_it2016_{day} M2
	JOIN BIGDN_M3_2 BIGDN_M3 ON M2.DN_ID = BIGDN_M3.ID
GROUP BY
	DAY, DAC_FAMILY, HOUR            
""", dbalchemy)

    values = df.to_numpy().tolist()
    nulls = []
    for h in range(24):
        for mw in [ 'conficker', 'modpack', 'necurs', 'pitou', 'suppobox', 'tofsee', 'virut' ]:
            nulls.append([day, h, mw] + np.zeros(len(features.split('\n'))).tolist())
    return pd.DataFrame(values + nulls, columns=df.columns).drop_duplicates(subset=['day','hour','dac_family'], keep='first')

pd.concat([ get(i) for i in range(10) ]).to_csv('days3.csv')

0
1
2
3
4
5
6
7
8
9


In [ ]:

	COUNT(*) AS QR,
[
    'IS_R IS FALSE', 'Q',
    'IS_R IS TRUE', 'R',
    'RCODE = 0', 'OK',
    'RCODE = 3', 'NX',
]
[
    'RN=1','DD',
    'RN_QR_RCODE=1','DDRCODE',
    'RN_MAC=1','DDMAC',
    'RN_MAC_QR_RCODE=1','DDRCODE_MAC',
]
[
	
]
	COUNT(DISTINCT DN_ID) AS UNIQUE_DN,
	COUNT(DISTINCT DN_ID) FILTER (WHERE bigdn_m3.regex_check) AS UNIQUE_VALIDS,
	COUNT(DISTINCT DN_ID) FILTER (WHERE RCODE=0) AS UNIQUE_OK,
	COUNT(DISTINCT DN_ID) FILTER (WHERE RCODE=3) AS UNIQUE_NXD,
	COUNT(DISTINCT DN_ID) FILTER (WHERE RCODE=3 AND NOT bigdn_m3.regex_check) AS UNIQUE_NXD_NOTVALIDS,





